# Oracle4SD Atlas: from public oracle evidence to a checked economic episode

**Data version:** `uma-public-rpc-case-v1` · **Review bundle:** `atlas-review-20260916` · **Cutoff:** 2026-06-30 23:59:59 UTC.

This tutorial is for colleagues new to blockchain. It reconstructs **one real UMA request** from five transactions across Polygon and Ethereum. It does not download or certify the entire 105-million-row candidate resource. A CPU runtime with Python 3.12, internet for setup and approximately 2 GB RAM is sufficient; no account key, wallet, GPU or payment is needed. Package installation usually takes longer than the case calculation. Measured local results are supplied with the repository.

## 1. Overview and navigation

1. Overview and setup (this section)
2. Data sources and metadata
3. Querying
4. Queried-data inspection
5. Processing
6. Processed-data inspection
7. Analysis
8. Technical validation, full-data instructions and troubleshooting

An **oracle** is a mechanism for supplying claims to a blockchain program. A **transaction** is an action recorded by the chain; a **receipt** records its execution and events. **Collateral/principal** is a participant's own money committed to the mechanism. A reward is an incremental gain, so a payment returning principal is not entirely a reward. A protocol can resolve a dispute without proving external factual truth.

We follow the mentor's eight-part template, using its repository-preserved version recorded in `metadata/template_provenance.json`. Collection methodology is compared with Luo et al. (NeurIPS 2024), §3.2: [official paper](https://proceedings.neurips.cc/paper_files/paper/2024/file/3205b048f9cc54b9f7963db0b0f52d53-Paper-Datasets_and_Benchmarks_Track.pdf). We use protocol-native evidence and explicit selection; no GoG code or explorer fraud labels are used as truth.

### Setup

The next cell clones an immutable **code commit**, then creates a fresh isolated Python environment. A commit is a fixed repository revision. A virtual environment keeps dependency versions separate from the notebook service. All scientific commands use that environment's Python. The checkout path, exact revision and package freeze are retained. Run cells in order with **Runtime → Run all**; repeated execution creates a new run directory. Setup downloads the repository and pinned packages; later bundled replay is offline.

In [ ]:
import csv, hashlib, json, os, shutil, subprocess, sys, tempfile, time
from pathlib import Path
from IPython.display import display, Markdown
REPO_URL = "https://github.com/virusLuke3/Oracle4SD-Atlas.git"
CODE_REF = "80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3"
DATA_VERSION = "uma-public-rpc-case-v1"
WORK = Path(tempfile.mkdtemp(prefix="oracle-atlas-tutorial-"))
REPO = WORK / "repository"
def command(args, cwd=None):
    child_env = os.environ.copy()
    child_env["MPLBACKEND"] = "Agg"  # Save figures without inheriting the host notebook backend.
    result = subprocess.run([str(a) for a in args], cwd=cwd, env=child_env, text=True, capture_output=True)
    if result.returncode:
        print(result.stdout); print(result.stderr)
        raise RuntimeError("Command failed; see diagnostic output above")
    if result.stderr:
        print(result.stderr)
    return result.stdout
command(["git", "clone", "--quiet", "--no-checkout", REPO_URL, REPO])
command(["git", "checkout", "--quiet", "--detach", CODE_REF], cwd=REPO)
assert command(["git", "rev-parse", "HEAD"], cwd=REPO).strip() == CODE_REF
base_python = sys.executable if sys.version_info[:2] == (3, 12) else shutil.which("python3.12")
if base_python is None:
    raise RuntimeError("Use a Python 3.12 runtime (or run locally with python3.12). No project stages have run.")
ENV = WORK / "case-environment"
command([base_python, "-m", "venv", ENV])
PYTHON = ENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
print(command([PYTHON, "-m", "pip", "install", "--disable-pip-version-check", "-r", REPO / "requirements.txt"])[-1200:])
RUN = REPO / "runs" / "tutorial"
RUN.mkdir(parents=True)
print({"code_commit": CODE_REF, "data_version": DATA_VERSION, "workspace": str(WORK)})

## 2. Data sources and metadata

The fixed source specification names four Polygon transactions and one Ethereum transaction. A **block** is an ordered collection of transactions; a block hash identifies its exact contents. The source register lists external origins, licenses and file hashes. **SHA256** detects changed file bytes; it cannot prove a claim is factually true.

Read [metadata/source_register.csv](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/metadata/source_register.csv), [metadata/acquisition_profiles.json](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/metadata/acquisition_profiles.json) and [docs/API_VERIFICATION.md](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/docs/API_VERIFICATION.md). Official [Ethereum JSON-RPC](https://ethereum.org/developers/docs/apis/json-rpc/) and [Polygon endpoint documentation](https://docs.polygon.technology/pos/reference/rpc-endpoints) define the relevant calls. The larger project uses additional chain-native APIs, described in the acquisition profiles. Expected: seven source files, two chains, five transactions and an unchanged data version.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text())
cfg = read_json(REPO / "configs/project.json")
spec = read_json(REPO / "data/data_source/uma_case/case_spec.json")
assert cfg["data_version"] == DATA_VERSION
assert len(cfg["sources"]) == 7
for chain, details in spec["chains"].items():
    print(chain, "chain ID", details["chain_id"], "transactions", len(details["transactions"]))
for source in cfg["sources"]:
    data = (REPO / source["path"]).read_bytes()
    assert hashlib.sha256(data).hexdigest() == source["sha256"]
    print(source["source_id"], len(data), "bytes; SHA256 verified")
print("Expected source transactions:", sum(len(c["transactions"]) for c in spec["chains"].values()))

## 3. Querying

For a dependable first run we acquire the frozen public evidence already in the checkout. This **bundled mode** replays actual queried receipts; it makes no new API requests. The next cell invokes [code/query_data/query.py](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/code/query_data/query.py) and records every acquired file.

Optional separate runs may use `--mode public-download` (two immutable Hugging Face snapshots, checksum verified) or `--mode live` (five transaction hashes, chain ID, receipts, headers and code). Live mode has bounded retries and may fail when public archive access is unavailable. No pagination is needed for these fixed transaction hashes. Full history requires block ranges or cursor pagination; see the profiles. Expected: mode `bundled`, seven checked files, five source transactions.

In [ ]:
def stage(folder, script, extra=()):
    return command([PYTHON, REPO / "code" / folder / script, "--config", REPO / "configs/project.json", "--run-dir", RUN, *extra], cwd=REPO)
stage("query_data", "query.py", ["--mode", "bundled"])
acquisition = read_json(RUN / "reports/acquisition.json")
assert acquisition["source_transactions"] == 5
assert all(f["frozen_hash_matches"] for f in acquisition["files"])
print({k: acquisition[k] for k in ["mode", "source_transactions", "network_scope", "failure_policy"]})

## 4. Queried-data inspection

Inspect the acquired receipts before deriving observations. A **log** is a contract event emitted in a transaction. A receipt may include logs unrelated to our selected economic flow; we do not count every log as a separate case. Expected: Polygon has four selected receipts and Ethereum one. The hash checks ensure the acquired copies match the registered inputs. Receipt success and exact canonical block matching are checked by the processing/QC stages.

In [ ]:
for chain in ["polygon", "ethereum"]:
    snapshot = read_json(RUN / "queried_data/uma_case/raw" / f"{chain}_rpc_snapshot.json")
    print(chain, {"chain_id": snapshot["chain_id"], "receipts": len(snapshot["receipts"]), "receipt_log_count": sum(len(r["logs"]) for r in snapshot["receipts"].values())})
    assert len(snapshot["receipts"]) == len(spec["chains"][chain]["transactions"])
for entry in acquisition["files"]:
    assert hashlib.sha256((RUN / entry["path"]).read_bytes()).hexdigest() == entry["sha256"]
print("Acquired bytes match the recorded checksums.")

## 5. Processing

The processor decodes event bytes using the **ABI** (the contract interface describing event fields), links a request to its dispute and adjudication, and applies the pinned contract settlement rule. It retains integer token units to avoid rounding and distinguishes returned principal from rewards. One **episode** is the linked lifecycle, not a single transaction.

[code/process_data/process.py](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/code/process_data/process.py) calls the existing real-case builder; [docs/METHODS.md](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/docs/METHODS.md) explains each transformation. Expected: one episode, nine decoded Polygon events, eight USDC.e transfers and the retained QC/provenance tables. This stage does not infer missing off-chain costs or independent truth.

In [ ]:
stage("process_data", "process.py")
processing = read_json(RUN / "reports/processing.json")
expected_counts = {"economic_episodes": 1, "source_events": 9, "token_flows": 8, "canonical_chain_qc": 30, "economic_link_qc": 13, "variable_provenance": 12, "lifecycle_timeline": 5, "economic_cashflows": 4, "abc_economic_measures": 28}
assert processing["counts"] == expected_counts
print(json.dumps(processing, indent=2))

## 6. Processed-data inspection

The episode has 85 fields; the project's larger common table has a different 49-column schema. The [metadata/data_dictionary.csv](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/metadata/data_dictionary.csv) defines every case field, unit, null policy and example. A JSON **null** means unavailable or not identified, not zero. The 28 core A/B/C measures are also distinct from the dictionary's 33 variables, which include five support variables.

The next cell inspects the focal participant and source linkage, lists unavailable fields and confirms that the episode is settled but independent truth is absent. The child tables retain event-level provenance so a reader can trace the result back to receipts.

In [ ]:
episode = read_json(RUN / "processed_data/economic_episode.json")
assert len(episode) == 85
print({key: episode[key] for key in ["episode_id", "actor_role", "terminal_outcome", "right_censored", "ground_truth_status", "source_transaction", "cross_chain_link_grade"]})
print("Unavailable fields:", [key for key, value in episode.items() if value is None])
print("Missingness explanation:", episode["missing_reason"])
with (RUN / "processed_data/lifecycle_timeline.csv").open() as handle:
    timeline = list(csv.DictReader(handle))
print("Lifecycle events:", len(timeline))
display(timeline)

## 7. Analysis

This is a descriptive reconstruction of one settlement. USDC.e uses six decimals, so 1,000,000 raw units equal one token. The winner receives **1,000 USDC.e = 750 returned principal + 250 token gain**. The configured request reward of 5 USDC.e is a separate quantity. Native gas is a different asset and is not subtracted from USDC.e without a timestamped conversion.

The next cell runs [code/analyze_data/analyze.py](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/code/analyze_data/analyze.py), displays the accounting components and renders a simple table. It fits no model and supports no population estimate or claim of total net profit.

In [ ]:
stage("analyze_data", "analyze.py")
summary = read_json(RUN / "analysis/summary.json")
scale = 10 ** episode["asset_decimals"]
principal = int(summary["principal_returned_raw"])
gain = int(summary["realized_payoff_raw"])
gross = int(summary["gross_payout_raw"])
assert (principal, gain, gross) == (750_000_000, 250_000_000, 1_000_000_000)
assert principal + gain == gross
display(Markdown("| Component | USDC.e |\n|---|---:|\n" + f"| Returned principal | {principal / scale:,.0f} |\n| Token gain | {gain / scale:,.0f} |\n| Gross transfer | {gross / scale:,.0f} |"))
print("Scope:", summary["scope"])

## 8. Technical validation

Validation asks whether these particular bytes, identities, links and monetary rules are internally consistent. [code/technical_validation/validate.py](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/code/technical_validation/validate.py) checks source hashes, the 85-field schema, a preserved earlier summary, original raw-derived assertions and child QC. Expected: **28 top-level checks pass, zero fail**, with 30 canonical and 13 economic/link checks retained underneath. Do not add these levels and call the total independent tests.

The next cell executes validation and displays the denominator for each check. A failed check raises an error rather than silently producing a successful-looking report. The frozen summary is a regression reference, not independent external truth.

In [ ]:
stage("technical_validation", "validate.py")
validation = read_json(RUN / "reports/validation.json")
assert validation["passed"] and validation["passed_checks"] == 28 and validation["failed_checks"] == 0
print({key: validation[key] for key in ["data_version", "passed_checks", "failed_checks", "not_evaluated"]})
display(validation["checks"])

### Independent rerun and provenance export

The next cell runs all four stages again in a new folder and compares the scientific summary. This checks deterministic reconstruction without reusing the first run's outputs. It also runs meaningful negative tests (changed source, wrong chain, missing receipt, removed log, payment mismatch, stale output and changed reference result). The export contains both run reports and an installed-package freeze. Runtime metadata is allowed to differ; the scientific summary must match. On Colab, use the Files sidebar to download the printed ZIP path.

In [ ]:
SECOND = REPO / "runs/independent-repeat"
print(command([PYTHON, REPO / "scripts/check_repository.py"], cwd=REPO))
print(command([PYTHON, REPO / "scripts/run_pipeline.py", "--output", SECOND], cwd=REPO))
assert read_json(RUN / "reconstruction/statistics/summary.json") == read_json(SECOND / "reconstruction/statistics/summary.json")
print(command([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=REPO))
(RUN / "reports/environment.freeze.txt").write_text(command([PYTHON, "-m", "pip", "freeze"]))
(RUN / "reports/tutorial_identity.json").write_text(json.dumps({"code_commit": CODE_REF, "data_version": DATA_VERSION, "review_bundle": cfg["review_bundle"], "cutoff": cfg["fixed_cutoff"], "runtime_kind": "hosted_colab" if "google.colab" in sys.modules else "local_jupyter_kernel", "scope": "one real case; independent repeat; not full candidate"}, indent=2))
export = shutil.make_archive(str(WORK / "oracle-atlas-case-evidence"), "zip", RUN)
print("Evidence ZIP:", export)
print("SUCCESS: one case reconstructed and checked twice; known full-core issues remain open.")

### Full dataset, resources and unresolved issues

The [metadata/full_data_inventory.csv](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/metadata/full_data_inventory.csv) gives exact hashes and row counts for 71 local candidate files (67 curated plus four applications; 15,474,890,321 bytes). The full immutable public archive and DOI are not available. The existing Hugging Face `1.5.0-draft` review subset is **not** the full core. The [docs/FULL_DATA.md](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/docs/FULL_DATA.md) instructions explain authorized-copy verification, isolated production staging, historical source registries, per-protocol commands and why current Pyth rolling state cannot reproduce the historical snapshot. Full collection needs archive endpoints, storage beyond the curated 15.5 GB and a production resource budget; it is not run by this notebook.

Known full-core problems remain in [metadata/issue_log.csv](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/metadata/issue_log.csv): dispute/slash labels, Tellor challenge-versus-report semantics, missing reward components, common-schema drift and missing timestamps. This case does not clear those issues. The Overleaf project and compiled manuscript PDF remain author-supplied handoff items; see [docs/HANDOFF.md](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/80745c2c9edcdd2c3b4f8bbf244d89751fe4d5f3/docs/HANDOFF.md).

### Troubleshooting

| Problem | Action |
|---|---|
| Setup says Python 3.12 missing | Select a compatible runtime or follow the local Python 3.12 instructions in the README |
| Package download fails | Restore network access and restart Run all; retain exact requirements |
| Source checksum differs | Re-fetch the pinned revision; never change expected hashes to force a pass |
| Output directory already exists | Run the setup cell again to allocate a new workspace |
| Live endpoint times out or cannot serve archive state | Retain the failed run; use a suitable endpoint or explicitly choose the bundled replay |
| Kernel was interrupted | Restart and Run all; do not continue with partial stage output |
| A scientific assertion fails | Read the stage stderr and validation JSON, keep the evidence and open a scoped issue |

### Review checklist

Confirm the code commit, data version and cutoff printed above; inspect the payment decomposition and null fields; verify the final checks; read the full-data limitations. The repository supplies an isolated local fresh-kernel execution record. That does not claim someone tested the Google-hosted Colab UI or independently reviewed the manuscript.